In [1]:
import glob
import os
import sys
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


sys.path.append('..')

from utils.experiment import (
    SimulationConfig,
    OptimizationResult,
    experiment_filename,
    save_experiment,
    load_experiment,
)


result_dirs = [
    '../Time Responsive Policy/Results/*.zip',
    '../Fixed Policy/Results/*.zip',
]

experiment_files = []

for result_dir in result_dirs:
    experiment_files.extend(glob.glob(result_dir))

max_workers = 24
os.makedirs('./Results', exist_ok=True)


In [2]:
experiment_df = pl.DataFrame(
    [],
    schema={
        'experiment_file': pl.Utf8,
        'initial_wealth': pl.Float64,
        'n_time_nodes': pl.Int64,
        'n_wealth_nodes': pl.Int64,
        'sampler': pl.Utf8,
    }
)


for experiment_file in experiment_files:
    experiment = load_experiment(experiment_file)
    result: OptimizationResult = experiment['result']
    config: SimulationConfig = experiment['config']

    policy = result.best_policy

    experiment_df = experiment_df.vstack(
        pl.DataFrame(
            {
                'experiment_file': [experiment_file],
                'initial_wealth': float(config.INITIAL_WEALTH),
                'n_time_nodes': [config.TIME_NODE_COUNT],
                'n_wealth_nodes': [config.WEALTH_NODE_COUNT],
                'sampler': [config.RETURN_SAMPLER],
            }
        )
    )

experiment_df.write_parquet('./Results/optimisation_experiments.parquet')

In [3]:
# Load best policy data
best_policy_df = pl.DataFrame(
    [],
    schema={
        'experiment_file': pl.Utf8,
        'time_node': pl.Float64,
        'wealth_node': pl.Float64,
        'cash': pl.Float64,
        'bonds': pl.Float64,
        'stocks': pl.Float64,
    }
)

for row in experiment_df.iter_rows(named=True):
    experiment_file = row['experiment_file']
    experiment = load_experiment(experiment_file)
    result: OptimizationResult = experiment['result']
    config: SimulationConfig = experiment['config']

    if config.TIME_NODE_COUNT > 1 and config.WEALTH_NODE_COUNT == 1:
        time_nodes = result.time_nodes
        # old versions of the code may not have time_nodes saved in the result, so we can construct them
        if time_nodes is None:
            time_nodes = np.linspace(0, config.SIMULATION_YEARS, config.TIME_NODE_COUNT)
        
        best_policy = result.best_policy
        policy_with_cash = np.zeros((best_policy.shape[0], best_policy.shape[1]+1))
        policy_with_cash[:, 1:] = best_policy
        # cash, bonds, stocks
        policy_with_cash[:, 0] = 1 - np.sum(best_policy, axis=1)
        
        best_policy_df = best_policy_df.vstack(
            pl.DataFrame(
                {
                    'experiment_file': [experiment_file] * len(time_nodes),
                    # cast to float 64
                    'time_node': time_nodes.astype(np.float64),
                    'wealth_node': [None] * len(time_nodes),
                    'cash': policy_with_cash[:, 0],
                    'bonds': policy_with_cash[:, 1],
                    'stocks': policy_with_cash[:, 2],
                }
            )
        )

    if config.TIME_NODE_COUNT == 1 and config.WEALTH_NODE_COUNT == 1:
        best_policy = result.best_policy
        policy_with_cash = np.zeros((1, best_policy.shape[0]+1))
        policy_with_cash[:, 1:] = best_policy
        # cash, bonds, stocks
        policy_with_cash[:, 0] = 1 - np.sum(best_policy, axis=0)

        best_policy_df = best_policy_df.vstack(
            pl.DataFrame(
                {
                    'experiment_file': [experiment_file] * len(policy_with_cash),
                    'time_node': [None] * len(policy_with_cash),
                    'wealth_node': [None] * len(policy_with_cash),
                    'cash': policy_with_cash[:, 0],
                    'bonds': policy_with_cash[:, 1],
                    'stocks': policy_with_cash[:, 2],
                }
            )
        )

best_policy_df.write_parquet('./Results/optimisation_best_policies.parquet')

c:\Users\CallumDavidson\Code\MADS\Code\Analysis\..\utils\experiment.py:130: UserWarning: Optional array 'time_nodes' is missing from this experiment archive. This is expected for older file versions; returning None.
  self._warn_missing_optional_array("time_nodes")


In [4]:

outcomes_schema = {
    'experiment_file': pl.Utf8,
    'bankruptcy_probability': pl.Float64,
    'bankruptcy_density': pl.Float64,
    'mean_terminal_wealth': pl.Float64,
    'std_terminal_wealth': pl.Float64,
    'median_terminal_wealth': pl.Float64,
    'p05_terminal_wealth': pl.Float64,
    'p10_terminal_wealth': pl.Float64,
    'p25_terminal_wealth': pl.Float64,
    'p75_terminal_wealth': pl.Float64,
    'p90_terminal_wealth': pl.Float64,
    'p95_terminal_wealth': pl.Float64,
}

def compute_outcome(experiment_file: str) -> dict:
    experiment = load_experiment(experiment_file)
    result: OptimizationResult = experiment['result']

    wealth = result.wealth_simulated
    final_wealth = wealth[:, -1]

    ruined = np.sum(wealth <= 0, axis=1)  # Count the number of times wealth is <= 0 for each simulation
    probability_ruined = np.mean(ruined > 0)  # Probability of being ruined at least once
    

    return {
        'experiment_file': experiment_file,
        'bankruptcy_probability': float(probability_ruined),
        'bankruptcy_density': float((wealth <= 0).mean()),
        'mean_terminal_wealth': float(final_wealth.mean()),
        'std_terminal_wealth': float(final_wealth.std()),
        'median_terminal_wealth': float(np.median(final_wealth)),
        'p05_terminal_wealth': float(np.percentile(final_wealth, 5)),
        'p10_terminal_wealth': float(np.percentile(final_wealth, 10)),
        'p25_terminal_wealth': float(np.percentile(final_wealth, 25)),
        'p75_terminal_wealth': float(np.percentile(final_wealth, 75)),
        'p90_terminal_wealth': float(np.percentile(final_wealth, 90)),
        'p95_terminal_wealth': float(np.percentile(final_wealth, 95)),
    }


outcome_rows = []

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(compute_outcome, experiment_file) for experiment_file in experiment_files]
    for future in tqdm.tqdm(
        as_completed(futures),
        total=len(futures),
        desc=f'Computing outcomes ({max_workers} threads)',
    ):
        outcome_rows.append(future.result())

outcomes_df = pl.DataFrame(outcome_rows, schema=outcomes_schema).sort('experiment_file')
outcomes_df.write_parquet('./Results/optimisation_outcomes.parquet')

Computing outcomes (24 threads):   0%|          | 0/198 [00:00<?, ?it/s]

Computing outcomes (24 threads): 100%|██████████| 198/198 [00:33<00:00,  5.89it/s]


In [14]:
timeseries_data_schema = {
    'experiment_file': pl.Utf8,
    'time_step': pl.Int64,
    'mean_wealth': pl.Float64,
    'std_wealth': pl.Float64,
    'median_wealth': pl.Float64,
    'p05_wealth': pl.Float64,
    'p10_wealth': pl.Float64,
    'p25_wealth': pl.Float64,
    'p75_wealth': pl.Float64,
    'p90_wealth': pl.Float64,
    'p95_wealth': pl.Float64,
    'bankruptcy_rate': pl.Float64,
}

def compute_timeseries(experiment_file: str) -> pl.DataFrame:
    experiment = load_experiment(experiment_file)
    result: OptimizationResult = experiment['result']

    wealth = result.wealth_simulated
    n_time_steps = wealth.shape[1]
    time_steps = np.arange(n_time_steps)

    mean_wealth = np.mean(wealth, axis=0)
    std_wealth = np.std(wealth, axis=0)
    bankruptcy_rate = np.mean(wealth <= 0, axis=0)

    # Compute all requested quantiles in one pass rather than multiple percentile scans.
    q05, q10, q25, q50, q75, q90, q95 = np.quantile(
        wealth,
        [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95],
        axis=0,
    )

    out = pl.DataFrame(
        {
            'experiment_file': [experiment_file] * n_time_steps,
            'time_step': time_steps,
            'mean_wealth': mean_wealth,
            'std_wealth': std_wealth,
            'median_wealth': q50,
            'p05_wealth': q05,
            'p10_wealth': q10,
            'p25_wealth': q25,
            'p75_wealth': q75,
            'p90_wealth': q90,
            'p95_wealth': q95,
            'bankruptcy_rate': bankruptcy_rate,
        }
    )
    return out

timeseries_rows: list[pl.DataFrame] = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(compute_timeseries, experiment_file) for experiment_file in experiment_files]
    for future in tqdm.tqdm(
        as_completed(futures),
        total=len(futures),
        desc=f'Computing timeseries ({max_workers} threads)',
    ):
        timeseries_rows.append(future.result())

timeseries_df = pl.concat(timeseries_rows).sort(['experiment_file', 'time_step'])
timeseries_df.write_parquet('./Results/optimisation_timeseries.parquet')

Computing timeseries (24 threads):   0%|          | 0/198 [00:00<?, ?it/s]

Computing timeseries (24 threads): 100%|██████████| 198/198 [01:08<00:00,  2.87it/s]


In [13]:
timeseries_df

experiment_file,time_step,mean_wealth,std_wealth,median_wealth,p05_wealth,p10_wealth,p25_wealth,p75_wealth,p90_wealth,p95_wealth,bankruptcy_rate
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""../Fixed Policy/Results\experi…",0,1e6,0.0,1e6,1e6,1e6,1e6,1e6,1e6,1e6,0.0
"""../Fixed Policy/Results\experi…",1,1.0488e6,220177.69911,1.0395e6,810991.556053,890806.290224,968421.617006,1.1221e6,1.2046e6,1.2643e6,0.0
"""../Fixed Policy/Results\experi…",2,1.1058e6,389218.254459,1.0712e6,713326.74661,847086.792133,957956.217834,1.2252e6,1.3719e6,1.4771e6,0.0
"""../Fixed Policy/Results\experi…",3,1.1676e6,554097.079031,1.1017e6,686315.799285,816327.02456,960229.742573,1.3099e6,1.5307e6,1.6912e6,0.000715
"""../Fixed Policy/Results\experi…",4,1.2414e6,889125.244436,1.1306e6,654991.755891,782422.248011,959843.598173,1.3911e6,1.7200e6,1.9223e6,0.000895
…,…,…,…,…,…,…,…,…,…,…,…
"""../Time Responsive Policy/Resu…",31,1.0385e6,922803.604817,808392.457686,37463.252607,155127.881496,406789.259508,1.4047e6,2.1851e6,2.7957e6,0.02561
"""../Time Responsive Policy/Resu…",32,1.0837e6,1.0001e6,828331.086067,16287.111199,137962.059778,402501.074434,1.4690e6,2.3149e6,2.9847e6,0.030305
"""../Time Responsive Policy/Resu…",33,1.1315e6,1.0829e6,847061.560431,6.8649e-12,120293.64081,397302.889981,1.5351e6,2.4542e6,3.1842e6,0.0355275
